# Urban Pulse Milestone 2: Predicting NYC Restaurant Inspection Risk

**Project title:** Tracking Urban Health Risk in NYC: Do Rodent Activity and Neighborhood Income Help Explain Restaurant Inspection Outcomes?

**Student:** Ramon Fernandez  
**Course:** Intro to Data Science  

## Research question
Do NYC ZIP codes with more rodent activity and lower median household income tend to have worse restaurant inspection outcomes?

## Datasets
1. **DOHMH New York City Restaurant Inspection Results** — restaurant inspection scores, grades, and violations.
2. **NYC Rodent Inspection Data** — rodent inspection counts and outcomes by ZIP code.
3. **ACS 2024 5-Year Median Household Income by ZCTA** — neighborhood income by ZIP Code Tabulation Area.

## Hypotheses from Milestone 1
- **Null hypothesis (H0):** After aggregating data by ZIP code, rodent inspection activity and neighborhood median household income have no statistically significant relationship with restaurant inspection outcomes in NYC.
- **Alternative hypothesis (HA):** After aggregating data by ZIP code, areas with higher rodent inspection activity and/or lower median household income have significantly worse restaurant inspection outcomes in NYC.

## Main target variable
The main outcome is **average restaurant inspection score by ZIP code**. In NYC restaurant inspections, a **lower score is better**, while a higher score means more/worse violations.

## 0. Setup
Run the notebook from top to bottom. It downloads public datasets directly, cleans them, merges them by ZIP/ZCTA, performs EDA, runs formal hypothesis tests, builds a predictive model, and saves final figures into an `output/` folder.

In [ ]:
# Core data science libraries
import os
import re
import textwrap
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

# Output folder for final visuals
os.makedirs('output', exist_ok=True)

## 1. Load the raw public datasets
The restaurant and rodent datasets come from NYC Open Data. The income dataset comes from the Census ACS API.

To keep the notebook reproducible and easy to run in Colab, the code pulls only the columns needed for this project.

In [ ]:
RESTAURANT_URL = "https://data.cityofnewyork.us/api/views/43nn-pn8j/rows.csv?accessType=DOWNLOAD"
RODENT_URL = "https://data.cityofnewyork.us/api/views/frm2-tra7/rows.csv?accessType=DOWNLOAD"
ACS_INCOME_URL = "https://api.census.gov/data/2024/acs/acs5?get=NAME,B19013_001E&for=zip%20code%20tabulation%20area:*"

restaurant_cols = [
    'CAMIS', 'DBA', 'BORO', 'ZIPCODE', 'CUISINE DESCRIPTION',
    'INSPECTION DATE', 'ACTION', 'VIOLATION CODE', 'CRITICAL FLAG',
    'SCORE', 'GRADE', 'INSPECTION TYPE'
]
rodent_cols = [
    'INSPECTION_TYPE', 'JOB_TICKET_OR_WORK_ORDER_ID', 'ZIP_CODE',
    'BOROUGH', 'INSPECTION_DATE', 'RESULT'
]

print('Downloading restaurant inspection data...')
restaurants_raw = pd.read_csv(RESTAURANT_URL, usecols=lambda c: c in restaurant_cols, low_memory=False)
print('Restaurant rows:', restaurants_raw.shape)

print('Downloading rodent inspection data...')
rodents_raw = pd.read_csv(RODENT_URL, usecols=lambda c: c in rodent_cols, low_memory=False)
print('Rodent rows:', rodents_raw.shape)

print('Downloading ACS income data...')
income_raw = pd.read_json(ACS_INCOME_URL)
income_raw.columns = income_raw.iloc[0]
income_raw = income_raw.iloc[1:].copy()
print('Income rows:', income_raw.shape)

## 2. Data cleaning and feature engineering

### Cleaning choices
- Standardize ZIP codes to 5-digit strings.
- Remove invalid restaurant inspection dates such as `1900-01-01`, which represent restaurants not yet inspected.
- Convert inspection scores to numeric values.
- Drop duplicate restaurant inspection rows caused by multiple violations being listed for the same inspection.
- Aggregate restaurant, rodent, and income data to the ZIP/ZCTA level.

In [ ]:
def clean_zip(value):
    """Return a 5-digit ZIP code string, or NaN if invalid."""
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    s = re.sub(r'\.0$', '', s)
    digits = re.sub(r'[^0-9]', '', s)
    if len(digits) >= 5:
        return digits[:5]
    return np.nan

# -----------------------------
# Clean restaurants
# -----------------------------
restaurants = restaurants_raw.copy()
restaurants.columns = restaurants.columns.str.lower().str.replace(' ', '_')

restaurants['zip'] = restaurants['zipcode'].apply(clean_zip)
restaurants['inspection_date'] = pd.to_datetime(restaurants['inspection_date'], errors='coerce')
restaurants['score'] = pd.to_numeric(restaurants['score'], errors='coerce')

restaurants = restaurants[
    restaurants['zip'].notna() &
    restaurants['score'].notna() &
    restaurants['inspection_date'].notna() &
    (restaurants['inspection_date'] > '1900-01-01')
].copy()

# Avoid over-counting inspections with multiple violation rows.
restaurant_inspections = restaurants.drop_duplicates(
    subset=['camis', 'inspection_date', 'score', 'grade', 'zip']
).copy()

# Create useful inspection indicators.
restaurant_inspections['poor_inspection'] = (restaurant_inspections['score'] >= 28).astype(int)  # roughly C-range or worse
restaurant_inspections['a_grade'] = (restaurant_inspections['grade'] == 'A').astype(int)
if 'critical_flag' in restaurant_inspections.columns:
    restaurant_inspections['critical_observed'] = restaurant_inspections['critical_flag'].eq('Critical').astype(int)
else:
    restaurant_inspections['critical_observed'] = 0

restaurant_zip = (
    restaurant_inspections
    .groupby('zip')
    .agg(
        avg_score=('score', 'mean'),
        median_score=('score', 'median'),
        max_score=('score', 'max'),
        poor_inspection_rate=('poor_inspection', 'mean'),
        a_grade_rate=('a_grade', 'mean'),
        restaurant_inspections=('score', 'count'),
        unique_restaurants=('camis', 'nunique')
    )
    .reset_index()
)

# -----------------------------
# Clean rodent inspections
# -----------------------------
rodents = rodents_raw.copy()
rodents.columns = rodents.columns.str.lower().str.replace(' ', '_')
rodents['zip'] = rodents['zip_code'].apply(clean_zip)
rodents['inspection_date'] = pd.to_datetime(rodents['inspection_date'], errors='coerce')
rodents = rodents[rodents['zip'].notna() & rodents['inspection_date'].notna()].copy()

rodents['active_rat_signs'] = rodents['result'].astype(str).str.contains('Active Rat Signs', case=False, na=False).astype(int)
rodents['problem_conditions'] = rodents['result'].astype(str).str.contains('Problem Conditions', case=False, na=False).astype(int)

rodent_zip = (
    rodents
    .groupby('zip')
    .agg(
        rodent_inspections=('result', 'count'),
        active_rat_signs=('active_rat_signs', 'sum'),
        problem_conditions=('problem_conditions', 'sum')
    )
    .reset_index()
)
rodent_zip['active_rat_sign_rate'] = rodent_zip['active_rat_signs'] / rodent_zip['rodent_inspections']
rodent_zip['problem_condition_rate'] = rodent_zip['problem_conditions'] / rodent_zip['rodent_inspections']

# -----------------------------
# Clean ACS income
# -----------------------------
income = income_raw.rename(columns={
    'B19013_001E': 'median_household_income',
    'zip code tabulation area': 'zip'
}).copy()
income['zip'] = income['zip'].apply(clean_zip)
income['median_household_income'] = pd.to_numeric(income['median_household_income'], errors='coerce')
income = income[income['zip'].notna() & income['median_household_income'].notna()].copy()

# -----------------------------
# Merge to ZIP level
# -----------------------------
df = restaurant_zip.merge(rodent_zip, on='zip', how='left').merge(income[['zip', 'median_household_income']], on='zip', how='left')

# Fill missing rodent metrics with 0 because a ZIP with restaurants but no rodent records has no observed rodent inspections in this dataset.
rodent_fill_cols = ['rodent_inspections', 'active_rat_signs', 'problem_conditions', 'active_rat_sign_rate', 'problem_condition_rate']
df[rodent_fill_cols] = df[rodent_fill_cols].fillna(0)

# Keep rows with usable income and enough restaurant data for stable ZIP-level estimates.
df = df[df['median_household_income'].notna()].copy()
df = df[df['restaurant_inspections'] >= 10].copy()

# Extra engineered features
# Add 1 before log to handle zero rodent inspections.
df['log_rodent_inspections'] = np.log1p(df['rodent_inspections'])
df['log_restaurant_inspections'] = np.log1p(df['restaurant_inspections'])
df['income_10k'] = df['median_household_income'] / 10000

print('Final ZIP-level modeling dataset:', df.shape)
display(df.head())

## 3. Exploratory Data Analysis: the “So What?” phase

This section looks for skew, outliers, relationships, and possible multicollinearity before modeling.

In [ ]:
summary_cols = [
    'avg_score', 'poor_inspection_rate', 'a_grade_rate', 'restaurant_inspections',
    'rodent_inspections', 'active_rat_sign_rate', 'problem_condition_rate',
    'median_household_income'
]

display(df[summary_cols].describe().T)

In [ ]:
# Feature distributions
for col in ['avg_score', 'median_household_income', 'rodent_inspections', 'active_rat_sign_rate', 'poor_inspection_rate']:
    plt.figure(figsize=(8, 5))
    plt.hist(df[col].dropna(), bins=25)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Number of ZIP codes')
    plt.tight_layout()
    plt.savefig(f'output/distribution_{col}.png', dpi=150)
    plt.show()

print('Interpretation:')
print('- If rodent_inspections is right-skewed, a few ZIP codes have extremely high rodent activity; that is why the model also uses log_rodent_inspections.')
print('- If avg_score is skewed or has outliers, RMSE may be sensitive to those high-risk ZIPs.')

In [ ]:
# Correlation heatmap without seaborn
corr_cols = [
    'avg_score', 'poor_inspection_rate', 'a_grade_rate', 'restaurant_inspections',
    'rodent_inspections', 'log_rodent_inspections', 'active_rat_sign_rate',
    'problem_condition_rate', 'median_household_income'
]
corr = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
im = plt.imshow(corr, aspect='auto')
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.index)), corr.index)
plt.title('Correlation Matrix: ZIP-Level Features')

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('output/correlation_heatmap.png', dpi=150)
plt.show()

print('Strongest correlations with avg_score:')
display(corr['avg_score'].sort_values(key=lambda s: s.abs(), ascending=False))

print('Interpretation:')
print('- The heatmap helps detect multicollinearity. If two predictors are very highly correlated, using both may not add much new information.')
print('- Features with stronger absolute correlation with avg_score are more promising predictors of inspection outcomes.')

In [ ]:
# Bivariate analysis: income vs inspection scores
plt.figure(figsize=(8, 5))
plt.scatter(df['median_household_income'], df['avg_score'], alpha=0.75)
plt.title('Bivariate Relationship: Income vs Average Restaurant Inspection Score')
plt.xlabel('Median Household Income by ZIP/ZCTA')
plt.ylabel('Average Restaurant Inspection Score (higher = worse)')
plt.tight_layout()
plt.savefig('output/income_vs_avg_score.png', dpi=150)
plt.show()

# Bivariate analysis: rodent inspections vs inspection scores
plt.figure(figsize=(8, 5))
plt.scatter(df['log_rodent_inspections'], df['avg_score'], alpha=0.75)
plt.title('Bivariate Relationship: Rodent Activity vs Average Restaurant Inspection Score')
plt.xlabel('log(1 + Rodent Inspections)')
plt.ylabel('Average Restaurant Inspection Score (higher = worse)')
plt.tight_layout()
plt.savefig('output/rodents_vs_avg_score.png', dpi=150)
plt.show()

In [ ]:
# Box plot: high vs low rodent activity ZIPs
median_rodents = df['rodent_inspections'].median()
df['rodent_activity_group'] = np.where(df['rodent_inspections'] > median_rodents, 'High rodent activity', 'Low rodent activity')

groups = [
    df.loc[df['rodent_activity_group'] == 'Low rodent activity', 'avg_score'],
    df.loc[df['rodent_activity_group'] == 'High rodent activity', 'avg_score']
]

plt.figure(figsize=(8, 5))
plt.boxplot(groups, labels=['Low rodent activity', 'High rodent activity'])
plt.title('Average Restaurant Score by Rodent Activity Group')
plt.ylabel('Average Restaurant Inspection Score (higher = worse)')
plt.tight_layout()
plt.savefig('output/boxplot_rodent_group_avg_score.png', dpi=150)
plt.show()

print(df.groupby('rodent_activity_group')['avg_score'].agg(['count', 'mean', 'median', 'std']))

## 4. Hypothesis Testing: statistical proof

We use two formal tests:

1. **Welch's t-test** comparing average inspection scores in high-rodent vs low-rodent ZIP codes.
2. **Pearson correlation test** between median household income and average inspection score.

A significance level of **α = 0.05** is used.

In [ ]:
alpha = 0.05

low_group = df.loc[df['rodent_activity_group'] == 'Low rodent activity', 'avg_score'].dropna()
high_group = df.loc[df['rodent_activity_group'] == 'High rodent activity', 'avg_score'].dropna()

t_stat, p_ttest = stats.ttest_ind(high_group, low_group, equal_var=False)

print('Welch t-test: High vs Low Rodent Activity ZIP Codes')
print(f'T-statistic: {t_stat:.4f}')
print(f'P-value: {p_ttest:.6f}')
print(f'High rodent group mean avg_score: {high_group.mean():.3f}')
print(f'Low rodent group mean avg_score: {low_group.mean():.3f}')

if p_ttest < alpha:
    print('Decision: Reject H0 for the rodent activity test.')
    print('Context: ZIP codes with higher rodent inspection activity have statistically different restaurant inspection scores than ZIPs with lower rodent activity.')
else:
    print('Decision: Fail to reject H0 for the rodent activity test.')
    print('Context: This sample does not provide enough statistical evidence that high- and low-rodent ZIP codes differ in average restaurant inspection score.')

In [ ]:
r_income, p_income = stats.pearsonr(df['median_household_income'], df['avg_score'])

print('Pearson correlation test: Income vs Average Restaurant Inspection Score')
print(f'Correlation coefficient r: {r_income:.4f}')
print(f'P-value: {p_income:.6f}')

if p_income < alpha:
    print('Decision: Reject H0 for the income relationship test.')
    print('Context: Median household income is statistically related to average restaurant inspection scores at the ZIP-code level.')
else:
    print('Decision: Fail to reject H0 for the income relationship test.')
    print('Context: This sample does not provide enough statistical evidence that income is linearly related to average restaurant inspection scores.')

## 5. Model Building: predicting average restaurant inspection score

This project uses **supervised regression** because the target variable is numeric: average restaurant inspection score by ZIP code.

Models used:
- Linear Regression as a simple baseline.
- Random Forest Regression as a non-linear model that can capture interactions between income and sanitation variables.

Evaluation metrics:
- **RMSE**: average prediction error in inspection-score points, with larger errors penalized more.
- **MAE**: average absolute prediction error.
- **R²**: proportion of target variation explained by the model.

In [ ]:
features = [
    'median_household_income',
    'log_rodent_inspections',
    'active_rat_sign_rate',
    'problem_condition_rate',
    'log_restaurant_inspections',
    'unique_restaurants'
]
target = 'avg_score'

model_df = df[features + [target, 'zip']].dropna().copy()

X = model_df[features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

linear_model = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

rf_model = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    min_samples_leaf=3
)

models = {
    'Linear Regression': linear_model,
    'Random Forest': rf_model
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    results.append({'model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2})

results_df = pd.DataFrame(results).sort_values('RMSE')
display(results_df)

best_model_name = results_df.iloc[0]['model']
print(f'Best model by RMSE: {best_model_name}')

In [ ]:
# Predicted vs actual plot for best model
best_model = models[best_model_name]
y_pred = best_model.predict(X_test)

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.8)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')
plt.title(f'Predicted vs Actual Average Inspection Score: {best_model_name}')
plt.xlabel('Actual avg_score')
plt.ylabel('Predicted avg_score')
plt.tight_layout()
plt.savefig('output/predicted_vs_actual.png', dpi=150)
plt.show()

comparison = pd.DataFrame({
    'actual_avg_score': y_test,
    'predicted_avg_score': y_pred,
    'error': y_pred - y_test
}).sort_values('error', key=lambda s: s.abs(), ascending=False)

display(comparison.head(10))

In [ ]:
# Feature importance / coefficients
if best_model_name == 'Random Forest':
    importance = pd.DataFrame({
        'feature': features,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
else:
    coefficients = best_model.named_steps['regressor'].coef_
    importance = pd.DataFrame({
        'feature': features,
        'importance': np.abs(coefficients),
        'coefficient': coefficients
    }).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
plt.barh(importance['feature'], importance['importance'])
plt.gca().invert_yaxis()
plt.title(f'Feature Importance: {best_model_name}')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('output/feature_importance.png', dpi=150)
plt.show()

display(importance)

## 6. Knowledge Discovery: the “Aha!” moment

Use the cells below to automatically generate a draft conclusion based on your actual results. After running the notebook, read the output and copy the strongest points into your final PDF report.

In [ ]:
# Automatically generate concise findings from the results.
strongest_corr = corr['avg_score'].drop('avg_score').abs().sort_values(ascending=False).index[0]
strongest_corr_value = corr.loc[strongest_corr, 'avg_score']

best_metrics = results_df.iloc[0]
top_feature = importance.iloc[0]['feature']

print('Milestone 2 Findings Draft')
print('--------------------------')
print(f'1. The strongest simple correlation with average restaurant inspection score was {strongest_corr} (r = {strongest_corr_value:.3f}).')
print(f'2. The rodent activity t-test p-value was {p_ttest:.6f}.')
print(f'3. The income correlation test p-value was {p_income:.6f}.')
print(f'4. The best predictive model was {best_metrics["model"]}, with RMSE = {best_metrics["RMSE"]:.3f}, MAE = {best_metrics["MAE"]:.3f}, and R² = {best_metrics["R2"]:.3f}.')
print(f'5. The most important model feature was {top_feature}.')
print()
print('Actionable insight draft:')
print(textwrap.fill(
    'NYC public-health agencies should use ZIP-level sanitation and socioeconomic indicators as a screening tool, not as a replacement for inspections. ZIP codes with elevated predicted inspection scores can be prioritized for outreach, prevention campaigns, and closer inspection scheduling. This is especially useful because the model combines restaurant outcomes, rodent conditions, and neighborhood income into one reproducible risk ranking.',
    width=100
))

In [ ]:
# Create a ZIP-level risk ranking table for the final report.
risk_df = df.copy()
risk_df['predicted_avg_score'] = best_model.predict(risk_df[features])
risk_df['risk_rank'] = risk_df['predicted_avg_score'].rank(ascending=False, method='dense').astype(int)

risk_ranking = risk_df[[
    'zip', 'risk_rank', 'predicted_avg_score', 'avg_score',
    'median_household_income', 'rodent_inspections',
    'active_rat_sign_rate', 'restaurant_inspections'
]].sort_values('risk_rank').head(15)

display(risk_ranking)
risk_ranking.to_csv('output/top_15_predicted_high_risk_zips.csv', index=False)
print('Saved output/top_15_predicted_high_risk_zips.csv')

## 7. Final report checklist

Your final PDF should include:

1. Project title, team member, and research question.
2. Data sources with working links.
3. Cleaning and merge explanation.
4. EDA plots with interpretations.
5. Formal hypothesis test with p-value and decision.
6. Model choice and evaluation metrics.
7. Feature importance and knowledge discovery.
8. Actionable urban policy insight.
9. Links to GitHub repository, Google Colab notebook, and video presentation.

The notebook saves major visuals to the `output/` folder so they can be used in your report or presentation.